# memrot presentation demo: audit -> ranked, LLM-mutated attack run

One straight-line walkthrough for a live demo: static audit -> severity-ranked,
LLM-mutated attack run against the real `genai-invest-agent-memory-stand` ->
the interactive HTML report. Each stage prints its own wall-clock time, and
the last cell prints the combined total -- exactly the number to watch during
a live run.

## Prerequisites

- The stand running: `cd ../genai-invest-agent-memory-stand && docker compose up -d`
  (needs ~30-60s to become healthy on a cold start).
- Fresh API keys for `cus` 1001-1005, minted headless (no browser SSO needed):
  ```bash
  cd ../genai-invest-agent-memory-stand
  docker compose exec -T agent-api python -c "
  from app.apikeys import generate_key
  from app.memory.mongo import MongoMemoryStore
  m = MongoMemoryStore()
  for cus in ['1001','1002','1003','1004','1005']:
      raw, rec = generate_key(cus, label='memrot-presentation')
      m.api_keys.create(rec)
      print(f'{cus}={raw}')
  "
  ```
  Paste the 5 printed `cus=key` lines into the `CRED_KEYS` dict in the setup
  cell below.
- An OpenRouter (or any OpenAI-compatible) API key for the judge + mutation
  LLM -- reads it straight from the stand's own `.env` (`OPENAI_API_KEY`) by
  default, reusing the exact key already configured there.
- This notebook was written but **deliberately not executed** -- run it live
  and watch the printed timings; that's the actual point of it.

In [ ]:
import os
import sys
import time
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "memrot").is_dir():
            return p
    raise RuntimeError("could not find the repo root (looked for a memrot/ directory)")


REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

STAND_ROOT = REPO_ROOT.parent / "genai-invest-agent-memory-stand"
from dotenv import load_dotenv
load_dotenv(STAND_ROOT / ".env")
os.environ["MEMROT_JUDGE_KEY"] = os.environ["OPENAI_API_KEY"]  # reused for judge + mutation LLM

# Paste the 5 lines printed by the key-minting command above.
CRED_KEYS = {
    "1001": "sk-genai-...",
    "1002": "sk-genai-...",
    "1003": "sk-genai-...",
    "1004": "sk-genai-...",
    "1005": "sk-genai-...",
}
for cus, key in CRED_KEYS.items():
    os.environ[f"MEMROT_CRED_CUS_{cus}"] = key

print("repo root:", REPO_ROOT)
print("stand root:", STAND_ROOT)

## Stage 1 -- static audit (timed)

Offline, no network calls to the stand: reads the portable, checked-in
manifest (`examples/genai_invest_stand.manifest.json`) and produces a JSON
report of findings with severity + `rule_id`, which the attack stage below
ranks its catalog by. This stage is fast (sub-second in every prior run) --
almost the entire pipeline's wall-clock lives in Stage 2.

In [ ]:
import subprocess

os.makedirs(".audit", exist_ok=True)
audit_t0 = time.time()
subprocess.run([
    sys.executable, "-m", "mcp_audit", "audit",
    "examples/genai_invest_stand.manifest.json",
    "--json", ".audit/stand.json", "--md", ".audit/stand.md",
], check=True)
audit_elapsed = time.time() - audit_t0
print(f"\nStage 1 (audit) wall-clock: {audit_elapsed:.1f}s")

## Stage 2 -- audit-ranked, LLM-mutated attack run against the real stand (timed)

The proven, full invest-bank catalog (`examples/genai_invest_stand.attack.config.json`
-- BAC/AUTH, MEM-01/03 cross-session semantic poisoning, framing diversity,
MEM-02 global-policy poisoning, benign controls), re-ordered by the audit's
own severity findings (`--audit-mode ranked`), then expanded with two
real, LLM-driven mutation techniques (`paraphrase`, `roleplay_framing` --
each seed variant gets rewritten by `openai/gpt-4o-mini`, tripling the
number of live target-agent turns: every original seed plus its two
mutated children). `--fancy` gives the full human-readable walkthrough
(banner, config panel, live model-connectivity check, audit summary before
the attack table, a live progress bar, results table, plain-language
summary) -- exactly what a live terminal demo shows, just captured here
too.

In [ ]:
attack_t0 = time.time()
subprocess.run([
    sys.executable, "-m", "memrot", "run",
    "--config", "examples/genai_invest_stand.attack.config.json",
    "--audit", ".audit/stand.json", "--audit-mode", "ranked",
    "--mutate", "paraphrase,roleplay_framing",
    "--mutation-base-url", "https://openrouter.ai/api/v1",
    "--mutation-model", "openai/gpt-4o-mini",
    "--mutation-api-key-env", "MEMROT_JUDGE_KEY",
    "--out", ".attack",
    "--report-html", ".attack/run.html",
    "--fancy",
])
attack_elapsed = time.time() - attack_t0
print(f"\nStage 2 (audit-ranked, LLM-mutated attack run) wall-clock: {attack_elapsed:.1f}s")

## Combined timing -- the number to watch live

The end-to-end number a presenter actually cares about: static audit +
severity-ranked, LLM-mutated attack run, back to back, no manual steps in
between.

In [ ]:
total_elapsed = audit_elapsed + attack_elapsed
print(f"Stage 1 (audit):                    {audit_elapsed:7.1f}s")
print(f"Stage 2 (ranked + LLM-mutated attack): {attack_elapsed:7.1f}s")
print(f"{'-' * 50}")
print(f"TOTAL (audit -> attack, end to end):  {total_elapsed:7.1f}s  ({total_elapsed / 60:.1f} min)")

## The report, inline

The same self-contained HTML dashboard written to `.attack/run.html` --
"Most Successful Attacks" chart up top (ranked by how many attacks actually
landed per category; hover or tab to a bar for the vulnerability it targets
and a real example of the payload that got through), no more Limitations
section cluttering the bottom, then the full per-axis ASR breakdown and
filterable per-variant results table below that.

In [ ]:
import html as html_lib
from IPython.display import display_html

report_html = Path(".attack/run.html").read_text(encoding="utf-8")
iframe = (
    f'<iframe srcdoc="{html_lib.escape(report_html)}" width="100%" height="900" '
    'style="border:1px solid #333;border-radius:8px;"></iframe>'
)
display_html(iframe, raw=True)